# 2 Feature Engineering & Exploration

Risk Factors: https://www.cdc.gov/heart-disease/risk-factors/index.html   

Several health conditions, your lifestyle, and your age and family history can increase your risk for heart disease. These are called risk factors. Key risk factors for heart disease include:

- High blood pressure
- High cholesterol
- Smoking

Some risk factors for heart disease cannot be controlled, such as your age or family history. But you can take steps to lower your risk by changing the factors you can control.
In addition

- Polynomial features (e.g., age², bmi²) --> To weigh these heavier
- Pulse Pressure = systolic_bp−diastolic_bp
- bmi * cholesterol: could capture the compounded risk of high weight and high lipids --> High BMI (overweight/obesity) is strongly associated with adverse lipid profiles, specifically increasing LDL ("bad") cholesterol and decreasing HDL ("good") cholesterol


If you use a linear model, these highly correlated features can cause instability. You might consider:
Mean Arterial Pressure (MAP): A weighted average of your two BP readings:
Mean Arterial Pressure: $MAP=diastolic + \frac{1}{3} (systolic- diastolic)$

In [1]:
# Load Data
import pandas as pd

df = pd.read_csv("data/raw/cardiovascular_risk_dataset.csv")
df.head()

,Patient_ID,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week,heart_disease_risk_score,risk_category
0,1,62,25.0,142,93,247,72,Never,11565,3,5.6,8.2,No,7,0.7,28.1,Medium
1,2,54,29.7,158,101,254,74,Current,4036,8,0.5,6.7,No,5,4.5,63.0,High
2,3,46,36.2,170,113,276,80,Current,3043,9,0.4,4.0,No,1,20.8,73.1,High
3,4,48,30.4,153,98,230,73,Former,5604,5,0.6,8.0,No,4,8.5,39.5,Medium
4,5,46,25.3,139,87,206,69,Current,7464,1,2.0,6.1,No,5,3.6,29.3,Medium


## 2.1 Data Cleaning

Before exploring which additional features could be created to consider the risk factors, we clean our original dataframe, based on what we found out in the EDA. As we did not find any missing values or duplicates, we will only drop columns. We will drop Patient_ID, as it is a unique identifier without any predictive value, and we will drop heart_disease_risk_score, as it directly encodes with our target (risk_category), which would cause leakage. 

In [2]:
# Drop Patient_ID and heart_disease_risk_score in a new dataframe so we do not change our original one 
df_clean = df.copy()

df_clean = df_clean.drop(columns=['Patient_ID', 'heart_disease_risk_score'])

print("Cleaned dataframe shape:", df_clean.shape)
print("Columns: ", df_clean.columns.tolist())

Cleaned dataframe shape: (5500, 15)
Columns:  ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol_mg_dl', 'resting_heart_rate', 'smoking_status', 'daily_steps', 'stress_level', 'physical_activity_hours_per_week', 'sleep_hours', 'family_history_heart_disease', 'diet_quality_score', 'alcohol_units_per_week', 'risk_category']


## 2.2 Feature Creation

As mentioned above, there are certain risk factors for getting a heart disease. While our dataset already provides some of these factors, by creating new features from the existing ones, we aim to better capture relationships that raw columns cannot express directly. We organise our engineered features into three categories: polynomial transformations, interaction features, and binned flag features.

**Polynomial transformations** (age_squared, bmi_squared): cardiovascular risk does not increase at a constant rate with age or BMI but accelerates at higher values. Squaring these variables allows models to reflect that non-linear behaviour.

**Interaction & ratio features** (pulse_pressure, map, bmi_x_age, bmi_x_cholesterol, cholesterol_age_ratio): these capture how two variables together create a stronger risk signal than either alone. For example, high BMI combined with high cholesterol represents a compounded metabolic risk, and high cholesterol at a younger age implies longer lifetime arterial exposure. Blood pressure variables are summarised into clinical metrics (Pulse Pressure and MAP) which are established markers of cardiovascular strain.

**Binary flag features** (hypertension_flag, obesity_flag, high_cholesterol_flag, inactivity_flag, sleep_deviation, alcohol_deviation): these encode clinically established thresholds directly, such as the WHO obesity threshold (BMI ≥ 30) or the hypertension boundary (systolic ≥ 140 or diastolic ≥ 90), giving models explicit and medically justified decision boundaries.

Not all engineered features will be used in the final models. Before modelling, we perform feature selection using Lasso to identify the most informative features, and check for redundancy introduced by our newly created variables before finalising our selection.

In [3]:
def feature_engineering(df):
    # Transform variables
    df['age_squared'] = df['age'] ** 2
    df['bmi_squared'] = df['bmi'] ** 2 

    # Add interactions
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    df['map'] = df['diastolic_bp'] + (df['systolic_bp'] - df['diastolic_bp']) / 3
    df['bmi_x_age'] = df['bmi'] * df['age']
    df['bmi_x_cholesterol'] = df['bmi'] * df['cholesterol_mg_dl']
    df['cholesterol_age_ratio'] = df['cholesterol_mg_dl'] / df['age']

    # Add Binary/ binned features 
    df['hypertension_flag'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)
    df['obesity_flag'] = (df['bmi'] >= 30).astype(int)
    df['high_cholesterol_flag'] = (df['cholesterol_mg_dl'] >= 240).astype(int)
    df['inactivity_flag'] = (df['daily_steps'] < 5000).astype(int)
    df['sleep_deviation'] = (df['sleep_hours'] - 7).abs()
    df['alcohol_deviation'] = df['alcohol_units_per_week'] - 14
    df['family_history_heart_disease'] = df['family_history_heart_disease'].map({'Yes': 1, 'No': 0})
    
    return df

## 2.3 Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split

df_all_features = feature_engineering(df_clean.copy())

# Define X/y
TARGET = "risk_category"
X = df_all_features.drop(columns=[TARGET])
y = df_all_features[TARGET]

# Split into training and testing set 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nClass distribution in training set:\n", y_train.value_counts(normalize=True))
print("\nClass distribution in test set:\n", y_test.value_counts(normalize=True))

Training set size: (4400, 27)
Test set size: (1100, 27)

Class distribution in training set:
 risk_category
Medium    0.404545
Low       0.338636
High      0.256818
Name: proportion, dtype: float64

Class distribution in test set:
 risk_category
Medium    0.421818
Low       0.316364
High      0.261818
Name: proportion, dtype: float64


In [6]:
# Encode target variable with intuitive ordinal mapping (Low=0, Medium=1, High=2)
risk_mapping = {'Low': 0, 'Medium': 1, 'High': 2}

y_train_encoded = y_train.map(risk_mapping)
y_test_encoded = y_test.map(risk_mapping)

# Check the mapping
for cls, idx in risk_mapping.items():
    print(f"{cls} → {idx}")

print("\nTraining target distribution:\n", y_train_encoded.value_counts().sort_index())

Low → 0
Medium → 1
High → 2

Training target distribution:
 risk_category
0    1490
1    1780
2    1130
Name: count, dtype: int64


## 2.4 Preprocessing

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

binary_cols = [
    'hypertension_flag', 'obesity_flag', 'high_cholesterol_flag',
    'inactivity_flag', 'family_history_heart_disease'
]

numerical_cols = [
    'age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol_mg_dl',
    'resting_heart_rate', 'daily_steps', 'stress_level',
    'physical_activity_hours_per_week', 'sleep_hours',
    'diet_quality_score', 'alcohol_units_per_week', 'age_squared',
    'bmi_squared', 'pulse_pressure', 'map', 'bmi_x_age', 'bmi_x_cholesterol',
    'cholesterol_age_ratio', 'sleep_deviation', 'alcohol_deviation'
]

categorical_cols = ['smoking_status']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('bin', 'passthrough', binary_cols)  # binary flags passed through unchanged
])

In [ ]:
# might remove and put in pipeline directly? 
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 2.5 Feature Selection with LassoCV

In [8]:
from sklearn.linear_model import LogisticRegressionCV

feature_names = (
    numerical_cols +
    list(preprocessor.named_transformers_['cat']
         .named_steps['encoder']
         .get_feature_names_out(categorical_cols))
)

logit_cv = LogisticRegressionCV(
    cv=5,
    penalty='l1',
    solver='saga',
    multi_class='multinomial',
    random_state=42,
    max_iter=10000
)
logit_cv.fit(X_train_processed, y_train_encoded)

# For multiclass, coef_ has shape (n_classes, n_features)
# Take the mean absolute coefficient across all classes
import numpy as np
coef_mean = np.mean(np.abs(logit_cv.coef_), axis=0)
coef_series = pd.Series(coef_mean, index=feature_names)
selected_features = coef_series[coef_series > 0].sort_values(ascending=False)

print(f"Features selected: {len(selected_features)}")
print(selected_features)

# Total features after engineering and encoding
total_features = X_train_processed.shape[1]
print(f"Total features after engineering and encoding: {total_features}")

/Users/max/Desktop/Machine_Learning_HeartDiseaseDetection/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1905: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Features selected: 27
smoking_status_Current              4.540688
bmi_x_age                           3.194258
smoking_status_Never                2.221280
systolic_bp                         2.124725
age                                 1.931327
family_history_heart_disease        1.861950
physical_activity_hours_per_week    1.715259
bmi_x_cholesterol                   1.315992
cholesterol_mg_dl                   1.314692
diet_quality_score                  1.197463
stress_level                        1.066929
age_squared                         0.580855
bmi_squared                         0.356562
obesity_flag                        0.195290
daily_steps                         0.173487
high_cholesterol_flag               0.136313
bmi                                 0.128287
inactivity_flag                     0.119990
cholesterol_age_ratio               0.109642
map                                 0.093139
sleep_hours                         0.089528
sleep_deviation                  

In [11]:
# Select features with mean absolute coefficient above 0.1 (threshold can be adjusted)
selected_features_final = coef_series[coef_series >= 0.1].sort_values(ascending=False)
print(selected_features_final)
print(f"Features selected: {len(selected_features_final)}")

# Get indices for subsetting processed arrays
selected_idx = [list(feature_names).index(f) for f in selected_features_final.index]
X_train_selected = X_train_processed[:, selected_idx]
X_test_selected = X_test_processed[:, selected_idx]

smoking_status_Current              4.540688
bmi_x_age                           3.194258
smoking_status_Never                2.221280
systolic_bp                         2.124725
age                                 1.931327
family_history_heart_disease        1.861950
physical_activity_hours_per_week    1.715259
bmi_x_cholesterol                   1.315992
cholesterol_mg_dl                   1.314692
diet_quality_score                  1.197463
stress_level                        1.066929
age_squared                         0.580855
bmi_squared                         0.356562
obesity_flag                        0.195290
daily_steps                         0.173487
high_cholesterol_flag               0.136313
bmi                                 0.128287
inactivity_flag                     0.119990
cholesterol_age_ratio               0.109642
dtype: float64
Features selected: 19


# 3 Modeling

## 3.1 Fit Baseline Logistic Regression (L1)

In [13]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score

y_pred_baseline = logit_cv.predict(X_test_processed)
y_prob_baseline = logit_cv.predict_proba(X_test_processed)

print("Logistic Regression (L1) Performance")
print("-"*40)
print(f"Accuracy:  {accuracy_score(y_test_encoded, y_pred_baseline):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_encoded, y_prob_baseline, multi_class='ovr'):.4f}")
print(f"F1:        {f1_score(y_test_encoded, y_pred_baseline, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test_encoded, y_pred_baseline, average='weighted'):.4f}")



# Compare train vs test performance
y_pred_train = logit_cv.predict(X_train_processed)

print("\nTrain vs Test Accuracy")
print("Train Accuracy:", accuracy_score(y_train_encoded, y_pred_train))
print("Test Accuracy: ", accuracy_score(y_test_encoded, y_pred_baseline))

Logistic Regression (L1) Performance
----------------------------------------
Accuracy:  0.9436
ROC-AUC:   0.9943
F1:        0.9437
Recall:    0.9436

Train vs Test Accuracy
Train Accuracy: 0.9440909090909091
Test Accuracy:  0.9436363636363636


## 3.2 Random Forest

## 3.3 XGBoost - Fit Model

**Note**:
What seemed to be an overfit on training data at first, turned out to be just a very good model, showing consistent performance on the train, as well as the test set. This suggests that our feautures are very strong in predicting the heart disease risk and the categories are clearly distinguishable for the model. 

In [17]:
import xgboost as xgb
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)

xgb.fit(X_train_selected, y_train_encoded)

y_pred_xgb = xgb.predict(X_test_selected)
y_prob_xgb = xgb.predict_proba(X_test_selected)

print("XGBoost Performance")
print("-" *40)
print(f"Accuracy:  {accuracy_score(y_test_encoded, y_pred_xgb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_encoded, y_prob_xgb, multi_class='ovr'):.4f}")
print(f"F1:        {f1_score(y_test_encoded, y_pred_xgb, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test_encoded, y_pred_xgb, average='weighted'):.4f}")

/Users/max/Desktop/Machine_Learning_HeartDiseaseDetection/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:36:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Performance
----------------------------------------
Accuracy:  0.9209
ROC-AUC:   0.9911
F1:        0.9211
Recall:    0.9209


### 3.3.1 XGBoost - Hyperparameter Tuning

In [32]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],          # shallower trees
    'learning_rate': [0.01, 0.05],   # slower learning
    'subsample': [0.5, 0.6, 0.7],    # more aggressive subsampling
    'colsample_bytree': [0.5, 0.6],  # fewer features per tree
    'min_child_weight': [5, 10, 20], # higher = more conservative splits
    'gamma': [0.5, 1, 2, 5],        # higher = more conservative splits
    'reg_alpha': [0.1, 0.5, 1, 5],  # L1 regularization
    'reg_lambda': [1, 2, 5, 10]     # L2 regularization
}

xgb = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss'
)

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train_selected, y_train_encoded)
print("Best params:", search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 4))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.6}
Best CV accuracy: 0.9189


In [33]:
best_xgb = search.best_estimator_

y_pred_tuned = best_xgb.predict(X_test_selected)
y_prob_tuned = best_xgb.predict_proba(X_test_selected)

print("Tuned XGBoost Performance")
print("-" *40)
print(f"Accuracy:  {accuracy_score(y_test_encoded, y_pred_tuned):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_encoded, y_prob_tuned, multi_class='ovr'):.4f}")
print(f"F1:        {f1_score(y_test_encoded, y_pred_tuned, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test_encoded, y_pred_tuned, average='weighted'):.4f}")

# Compare train vs test performance
print("\nTrain vs Test Accuracy for Tuned XGBoost")
y_pred_train_xgb = best_xgb.predict(X_train_selected)
print("Train Accuracy:", accuracy_score(y_train_encoded, y_pred_train_xgb))
print("Test Accuracy: ", accuracy_score(y_test_encoded, y_pred_tuned))

Tuned XGBoost Performance
----------------------------------------
Accuracy:  0.9291
ROC-AUC:   0.9906
F1:        0.9293
Recall:    0.9291

Train vs Test Accuracy for Tuned XGBoost
Train Accuracy: 0.9561363636363637
Test Accuracy:  0.9290909090909091


In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results = pd.DataFrame({
    'Model': [
        'Logistic Regression (L1)',
        'XGBoost (untuned)',
        'XGBoost (tuned)'
    ],
    'Accuracy':  [0.9436, 0.9209, 0.9291],
    'ROC-AUC':   [0.9943, 0.9911, 0.9906],
    'F1':        [0.9437, 0.9211, 0.9293],
    'Recall':    [0.9436, 0.9209, 0.9291],
})

print(results.to_string(index=False))

                   Model  Accuracy  ROC-AUC     F1  Recall
Logistic Regression (L1)    0.9436   0.9943 0.9437  0.9436
       XGBoost (untuned)    0.9209   0.9911 0.9211  0.9209
         XGBoost (tuned)    0.9291   0.9906 0.9293  0.9291


**Note**:

Baseline: Logistic Regression with L1 penalty
Advanced: XGBoost (gradient boosted trees)

Key Findings:
- Logistic Regression achieved 94.36% accuracy with a near-zero train/test gap (0.04%) indicating perfect generalization.

- XGBoost initially overfit significantly (train/test gap of 6.5%), requiring two rounds of regularization tuning to reduce
  the gap to an acceptable 2.7%.

- Even after tuning, XGBoost (92.91%) could not surpass the baseline (94.36%).

Conclusion:
Logistic regression (L1) is the superior model for this dataset. Heart disease risk features (BMI, cholesterol, blood pressure, and smoking status) have strong linear relationships with the target. This means added complexity from XGBoost does not improve generalization and initially decreases it.

In [39]:
from sklearn.metrics import classification_report
print(classification_report(y_test_encoded, y_pred_baseline, target_names=le.classes_))

              precision    recall  f1-score   support

        High       0.95      0.95      0.95       288
         Low       0.95      0.95      0.95       348
      Medium       0.93      0.94      0.93       464

    accuracy                           0.94      1100
   macro avg       0.95      0.95      0.95      1100
weighted avg       0.94      0.94      0.94      1100



# 4 Evaluation